In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from plotly.subplots import make_subplots
import torch
import umap
import plotly.graph_objects as go
from scipy.interpolate import griddata
from lightning_models.lightning_conformal_mlp import LightningConformalClassifier
import itertools
import plotly.graph_objects as go
from datamodules.bray_data_module import BrayDataModule
from datamodules.image_feature_data_module import ImageFeatureDataModule
from lightning_models.lightning_diffusion_mlp import LightningDiffusionClassifier

In [2]:
datamodule = BrayDataModule()
# datamodule = ImageFeatureDataModule(
#     split_col='hier_split_0',
#     data_csv_path='/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/data/bray_dino/bray_dino_complete.csv',
#     mask_uncertain= False,
#     validation_split_val="cal"
# )
datamodule.setup("fit")
datamodule

Bray DM init
/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/data/gigadb//gigadb.csv
Bray DM setup
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 35639 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 1069170
Total 1s (positive): 43327 (4.05%)
Total 0s (negative): 216227 (20.22%)
Total uncertain: 809616 (75.72%)
Positive-to-Negative ratio: 0.2004
Dataset loaded with 35639 samples
Feature dimension: 4643
Target dimension: 30
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 11371 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 341130
Total 1s (positive): 13659 (4.00%)
Total 0s (negative): 68427 (20.06%)
Total uncertai

In [3]:
 # get the training dataset
train_dataset = datamodule.train_dataloader()

# collect all features into numpy array
X = []
for batch in train_dataset:
    features, _ , _= batch  # (x, y)
    X.append(features)

/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [4]:
datamodule.setup("test")
 # get the training dataset
test_dataset = datamodule.test_dataloader()

# Get test set from dataloader
X_test, y_test, mask_test = [], [], []

for x, y, mask in datamodule.val_dataloader():
    if x.ndim > 2:
        x = x.view(x.size(0), -1)
    X_test.append(x)
    y_test.append(y)
    mask_test.append(mask)

X_test = torch.cat(X_test, dim=0).numpy()
y_test = torch.cat(y_test, dim=0).numpy()
mask_test = torch.cat(mask_test, dim=0).numpy()


Bray DM setup
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 6642 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 199260
Total 1s (positive): 8061 (4.05%)
Total 0s (negative): 37452 (18.80%)
Total uncertain: 153747 (77.16%)
Positive-to-Negative ratio: 0.2152
Dataset loaded with 6642 samples
Feature dimension: 4643
Target dimension: 30
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 11371 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 341130
Total 1s (positive): 13659 (4.00%)
Total 0s (negative): 68427 (20.06%)
Total uncertain: 259044 (75.94%)
Positive-to-Negative ratio: 0.1996
Dataset loaded with 11371 samples
Featur

In [5]:
X = torch.cat(X, dim=0).numpy()
clf = IsolationForest(random_state=42).fit(X)

In [6]:
# Predict with IsolationForest
y_pred = clf.predict(X_test)   # -1 = anomaly (OOD), 1 = normal (ID)

# Split into ID and OOD
in_distribution = {
    "X": X_test[y_pred == 1],
    "y": y_test[y_pred == 1],
    "mask": mask_test[y_pred == 1]
}

out_of_distribution = {
    "X": X_test[y_pred == -1],
    "y": y_test[y_pred == -1],
    "mask": mask_test[y_pred == -1]
}

print("In-distribution samples:", in_distribution["X"].shape[0])
print("Out-of-distribution samples:", out_of_distribution["X"].shape[0])

In-distribution samples: 10807
Out-of-distribution samples: 564


In [7]:
conf_ckpt_path = "/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/gridsearch_final/bray/mlp/md/1e-3/last.ckpt"
conf_model = LightningConformalClassifier(
    size="md",
    num_classes=30,
    embedding_dim=4643,
    loss_pos_weight=5.21,
    alpha=0.05,
    lr=1e-5,
    residual=True,
    activation_fn=torch.nn.ReLU,
)
conf_model.load_state_dict(
    torch.load(conf_ckpt_path, weights_only=True, map_location=torch.device("mps"))[
        "state_dict"
    ]
)
conf_model.eval()

LightningConformalClassifier(
  (model): MLPClassifierBBBC(
    (input_proj): Linear(in_features=4643, out_features=768, bias=True)
    (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (fc1): Linear(in_features=768, out_features=512, bias=True)
    (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (fc2): Linear(in_features=512, out_features=256, bias=True)
    (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc3): Linear(in_features=256, out_features=128, bias=True)
    (ln3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (fc4): Linear(in_features=128, out_features=30, bias=True)
    (res_proj1): Linear(in_features=768, out_features=512, bias=True)
    (res_proj2): Linear(in_features=512, out_features=256, bias=True)
    (res_proj3): Linear(in_features=256, out_features=128, bias=True)
    (activation): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (dropout2): Dropout(p=0.44999999999999996, inplace=False)
  )

In [8]:
diff_ckpt_path = "/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/gridsearch_final/bray/diffusion/md/1e-3/last.ckpt"
diff_model = LightningDiffusionClassifier(
    size="md",
    num_classes=30,
    embedding_dim=4643,
    loss_pos_weight=5.21,
    alpha=0.05,
    lr=1e-3,
    residual=True,
    activation_fn=torch.nn.ReLU,
)
diff_model.load_state_dict(
    torch.load(diff_ckpt_path, weights_only=True, map_location=torch.device("mps"))[
        "state_dict"
    ]
)
diff_model.eval()

LightningDiffusionClassifier(
  (model): DiffusionClassifierMd(
    (input_proj): Linear(in_features=4674, out_features=768, bias=True)
    (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (fc1): Linear(in_features=768, out_features=512, bias=True)
    (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (fc2): Linear(in_features=512, out_features=256, bias=True)
    (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc3): Linear(in_features=256, out_features=128, bias=True)
    (ln3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (fc4): Linear(in_features=128, out_features=30, bias=True)
    (res_proj1): Linear(in_features=768, out_features=512, bias=True)
    (res_proj2): Linear(in_features=512, out_features=256, bias=True)
    (res_proj3): Linear(in_features=256, out_features=128, bias=True)
    (activation): ReLU()
    (sigmoid): Sigmoid()
    (dropout1): Dropout(p=0.3, inplace=False)
    (dropout2): Dropout(p=0.44999999

In [9]:
preds_conf = []
for item in in_distribution["X"]:
    pred = conf_model.forward(torch.from_numpy(item))
    preds_conf.append(pred)

In [10]:
preds_array = np.array([pred.detach().numpy() if torch.is_tensor(pred) else pred for pred in preds_conf])

# preds_array should now have shape (n_samples, 30)
print(f"Predictions array shape: {preds_array.shape}")

# Create subplots - 6 rows x 5 columns for 30 histograms
fig = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

# Create histogram for each column
for i in range(30):
    row = i // 5 + 1  # Calculate row (1-indexed)
    col = i % 5 + 1   # Calculate column (1-indexed)

    # Add histogram trace
    fig.add_trace(
        go.Histogram(
            x=preds_array[:, i],
            nbinsx=20,  # Number of bins
            name=f'Col {i}',
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title_text="MLP, In distribution",
    title_x=0.5,
    height=1200,  # Adjust height as needed
    showlegend=False
)

# Update x and y axis labels
fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Count")

# Show the plot
fig.show()
fig.write_image('/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/historgrams/MLP_IN_BRAY.png', height=1200, width=1600)

Predictions array shape: (10807, 30)


In [11]:
preds_conf = []
for item in out_of_distribution["X"]:
    pred = conf_model.forward(torch.from_numpy(item))
    preds_conf.append(pred)

In [12]:
preds_array = np.array([pred.detach().numpy() if torch.is_tensor(pred) else pred for pred in preds_conf])

# preds_array should now have shape (n_samples, 30)
print(f"Predictions array shape: {preds_array.shape}")

# Create subplots - 6 rows x 5 columns for 30 histograms
fig = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

# Create histogram for each column
for i in range(30):
    row = i // 5 + 1  # Calculate row (1-indexed)
    col = i % 5 + 1   # Calculate column (1-indexed)

    # Add histogram trace
    fig.add_trace(
        go.Histogram(
            x=preds_array[:, i],
            nbinsx=20,  # Number of bins
            name=f'Col {i}',
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title_text="MLP, Out of distribution",
    title_x=0.5,
    height=1200,  # Adjust height as needed
    showlegend=False
)

# Update x and y axis labels
fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Count")

# Show the plot
fig.show()
fig.write_image('/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/historgrams/MLP_OUT_BRAY.png', height=1200, width=1600)

Predictions array shape: (564, 30)


In [13]:
preds_diff = []
for X in out_of_distribution["X"]:
    # Convert X to tensor and add batch dimension
    X_tensor = torch.from_numpy(X).float().unsqueeze(0)  # Shape: (1, 4643)
    # Create labels tensor with batch dimension
    labels_tensor =  torch.randn(1, 30)  # Shape: (1, 30)
    
    pred = diff_model.forward(X_tensor, labels_tensor)
    preds_diff.append(pred.detach().numpy()[0])

In [14]:
preds_array = np.array([pred.detach().numpy() if torch.is_tensor(pred) else pred for pred in preds_diff])

# preds_array should now have shape (n_samples, 30)
print(f"Predictions array shape: {preds_array.shape}")

# Create subplots - 6 rows x 5 columns for 30 histograms
fig = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

# Create histogram for each column
for i in range(30):
    row = i // 5 + 1  # Calculate row (1-indexed)
    col = i % 5 + 1   # Calculate column (1-indexed)

    # Add histogram trace
    fig.add_trace(
        go.Histogram(
            x=preds_array[:, i],
            nbinsx=20,  # Number of bins
            name=f'Col {i}',
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title_text="Diffusion, Out of distribution",
    title_x=0.5,
    height=1200,  # Adjust height as needed
    showlegend=False
)

# Update x and y axis labels
fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Count")

# Show the plot
fig.show()
fig.write_image('/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/historgrams/DIFF_OUT_BRAY.png', height=1200, width=1600)

Predictions array shape: (564, 30)


In [15]:
preds_diff = []
for X in in_distribution["X"]:
    # Convert X to tensor and add batch dimension
    X_tensor = torch.from_numpy(X).float().unsqueeze(0)  # Shape: (1, 4643)
    # Create labels tensor with batch dimension
    labels_tensor =  torch.randn(1, 30)  # Shape: (1, 30)

    pred = diff_model.forward(X_tensor, labels_tensor)
    preds_diff.append(pred.detach().numpy()[0])

In [16]:
preds_array = np.array([pred.detach().numpy() if torch.is_tensor(pred) else pred for pred in preds_diff])

# preds_array should now have shape (n_samples, 30)
print(f"Predictions array shape: {preds_array.shape}")

# Create subplots - 6 rows x 5 columns for 30 histograms
fig = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

# Create histogram for each column
for i in range(30):
    row = i // 5 + 1  # Calculate row (1-indexed)
    col = i % 5 + 1   # Calculate column (1-indexed)

    # Add histogram trace
    fig.add_trace(
        go.Histogram(
            x=preds_array[:, i],
            nbinsx=20,  # Number of bins
            name=f'Col {i}',
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title_text="Diffusion, in distribution",
    title_x=0.5,
    height=1200,  # Adjust height as needed
    showlegend=False
)

# Update x and y axis labels
fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Count")

# Show the plot
fig.show()
fig.write_image('/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/historgrams/DIFF_IN_BRAY.png', height=1200, width=1600)

Predictions array shape: (10807, 30)


In [17]:
# Collect all predictions for overlaid curves
import numpy as np
from scipy.stats import gaussian_kde

# Get MLP predictions for in-distribution
mlp_in_preds = []
for item in in_distribution["X"]:
    pred = conf_model.forward(torch.from_numpy(item))
    mlp_in_preds.append(pred.detach().numpy())
mlp_in_preds = np.array(mlp_in_preds)

# Get MLP predictions for out-of-distribution
mlp_out_preds = []
for item in out_of_distribution["X"]:
    pred = conf_model.forward(torch.from_numpy(item))
    mlp_out_preds.append(pred.detach().numpy())
mlp_out_preds = np.array(mlp_out_preds)

# Get diffusion predictions for in-distribution
diff_in_preds = []
for X in in_distribution["X"]:
    X_tensor = torch.from_numpy(X).float().unsqueeze(0)
    labels_tensor = torch.randn(1, 30)
    pred = diff_model.forward(X_tensor, labels_tensor)
    diff_in_preds.append(pred.detach().numpy()[0])
diff_in_preds = np.array(diff_in_preds)

# Get diffusion predictions for out-of-distribution
diff_out_preds = []
for X in out_of_distribution["X"]:
    X_tensor = torch.from_numpy(X).float().unsqueeze(0)
    labels_tensor = torch.randn(1, 30)
    pred = diff_model.forward(X_tensor, labels_tensor)
    diff_out_preds.append(pred.detach().numpy()[0])
diff_out_preds = np.array(diff_out_preds)

print(f"MLP in-dist shape: {mlp_in_preds.shape}")
print(f"MLP out-dist shape: {mlp_out_preds.shape}")
print(f"Diffusion in-dist shape: {diff_in_preds.shape}")
print(f"Diffusion out-dist shape: {diff_out_preds.shape}")

MLP in-dist shape: (10807, 30)
MLP out-dist shape: (564, 30)
Diffusion in-dist shape: (10807, 30)
Diffusion out-dist shape: (564, 30)


In [18]:
# Create separate plots for in-distribution and out-of-distribution
from scipy.stats import gaussian_kde

# ==== IN-DISTRIBUTION PLOT ====
fig_in = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

for i in range(30):
    row = i // 5 + 1
    col = i % 5 + 1
    
    # Get data for this class (in-distribution only)
    mlp_data = mlp_in_preds[:, i]
    diff_data = diff_in_preds[:, i]
    
    # Create common x range
    all_data = np.concatenate([mlp_data, diff_data])
    x_min, x_max = all_data.min(), all_data.max()
    x_range = np.linspace(x_min, x_max, 100)
    
    try:
        # MLP (blue solid)
        kde_mlp = gaussian_kde(mlp_data)
        y_mlp = kde_mlp(x_range)
        fig_in.add_trace(
            go.Scatter(x=x_range, y=y_mlp, mode='lines', name='MLP',
                      line=dict(color='blue', width=2), showlegend=(i==0)),
            row=row, col=col
        )
        
        # Diffusion (red dashed)
        kde_diff = gaussian_kde(diff_data)
        y_diff = kde_diff(x_range)
        fig_in.add_trace(
            go.Scatter(x=x_range, y=y_diff, mode='lines', name='Diffusion',
                      line=dict(color='red', width=2, dash='dash'), showlegend=(i==0)),
            row=row, col=col
        )
        
    except Exception as e:
        print(f"Error processing in-dist class {i}: {e}")

fig_in.update_layout(
    title_text="In-Distribution: MLP vs Diffusion Prediction Densities",
    title_x=0.5,
    height=1200,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_in.update_xaxes(title_text="Prediction Value")
fig_in.update_yaxes(title_text="Density")
fig_in.show()

print("=" * 50)

# ==== OUT-OF-DISTRIBUTION PLOT ====
fig_out = make_subplots(
    rows=6, cols=5,
    subplot_titles=[f'Class {i}' for i in range(30)],
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

for i in range(30):
    row = i // 5 + 1
    col = i % 5 + 1
    
    # Get data for this class (out-of-distribution only)
    mlp_data = mlp_out_preds[:, i]
    diff_data = diff_out_preds[:, i]
    
    # Create common x range
    all_data = np.concatenate([mlp_data, diff_data])
    x_min, x_max = all_data.min(), all_data.max()
    x_range = np.linspace(x_min, x_max, 100)
    
    try:
        # MLP (blue solid)
        kde_mlp = gaussian_kde(mlp_data)
        y_mlp = kde_mlp(x_range)
        fig_out.add_trace(
            go.Scatter(x=x_range, y=y_mlp, mode='lines', name='MLP',
                      line=dict(color='blue', width=2), showlegend=(i==0)),
            row=row, col=col
        )
        
        # Diffusion (red dashed)
        kde_diff = gaussian_kde(diff_data)
        y_diff = kde_diff(x_range)
        fig_out.add_trace(
            go.Scatter(x=x_range, y=y_diff, mode='lines', name='Diffusion',
                      line=dict(color='red', width=2, dash='dash'), showlegend=(i==0)),
            row=row, col=col
        )
        
    except Exception as e:
        print(f"Error processing out-dist class {i}: {e}")

fig_out.update_layout(
    title_text="Out-of-Distribution: MLP vs Diffusion Prediction Densities",
    title_x=0.5,
    height=1200,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_out.update_xaxes(title_text="Prediction Value")
fig_out.update_yaxes(title_text="Density")
fig_out.show()

In [19]:
no

NameError: name 'no' is not defined